In [ ]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
!git clone https://github.com/BennyTMT/LLMsForTimeSeries.git
%cd /kaggle/working/LLMsForTimeSeries/PAttn

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib tqdm statsmodels

In [ ]:
!find /kaggle/input -name "*.csv"

In [ ]:
!mkdir -p dataset/ETT

!cp /kaggle/input/datasets/trongmanh/ettm2-csv/ETTm2.csv \
dataset/ETT/ETTm2.csv

In [ ]:
!ls dataset/ETT

In [ ]:
import pandas as pd

raw_path = "dataset/ETT/ETTm2.csv"
out_path = "dataset/ETT/ETTm2_pattn.csv"

df = pd.read_csv(raw_path)

# đổi tên cột thời gian về "date"
if "date" not in df.columns:
    for col in ["timestamp", "time", "datetime", "open_time"]:
        if col in df.columns:
            df = df.rename(columns={col: "date"})
            break

df["date"] = pd.to_datetime(df["date"])

# ETTm2 thường chỉ có:
# date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT

cols = [
    "date",
    "HUFL",
    "HULL",
    "MUFL",
    "MULL",
    "LUFL",
    "LULL",
    "OT"
]

df = df[cols].dropna().sort_values("date")

for c in cols:
    if c != "date":
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna()

df.to_csv(out_path, index=False)

print(df.shape)
df.head()

In [ ]:
!sed -i 's/return mse, mae/return mse, mae, preds, trues/g' utils/tools.py

In [ ]:
path = "main.py"

with open(path, "r") as f:
    text = f.read()

old = """mse, mae = test(model, test_data, test_loader, args, device, ii)
    mses.append(round(mse,5))
    maes.append(round(mae,5))"""

new = """mse, mae, preds, trues = test(model, test_data, test_loader, args, device, ii)

    np.save("pattn_pred.npy", preds)
    np.save("pattn_true.npy", trues)

    print("Prediction files saved: pattn_pred.npy, pattn_true.npy")

    mses.append(round(mse, 5))
    maes.append(round(mae, 5))"""

text = text.replace(old, new)

with open(path, "w") as f:
    f.write(text)

In [ ]:
!sed -i 's/np.Inf/np.inf/g' utils/tools.py

In [ ]:
!python main.py \
  --model_id ETTm2_PAttn_full_sl512_pl24_u2 \
  --model PAttn \
  --root_path ./dataset/ETT/ \
  --data_path ETTm2_pattn.csv \
  --data custom \
  --features M \
  --target OT \
  --seq_len 512 \
  --label_len 0 \
  --pred_len 24 \
  --enc_in 7 \
  --c_out 7 \
  --freq 0 \
  --patch_size 16 \
  --stride 8 \
  --learning_rate 0.0005 \
  --train_epochs 10 \
  --decay_fac 0.5 \
  --d_model 768 \
  --n_heads 4 \
  --d_ff 768 \
  --dropout 0.2 \
  --patience 10 \
  --percent 100 \
  --gpt_layer 6 \
  --itr 1 \
  --tmax 50 \
  --cos 1 \
  --method PAttn \
  --gpu_loc 0 \
  --is_gpt 1 \
  --save_file_name pattn_ettm2_results_pl24.txt | tee pattn_ettm2_train_pl24.log

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load prediction
pred = np.load("pattn_pred.npy")
true = np.load("pattn_true.npy")

# Load ETTm2 dataset
df = pd.read_csv("dataset/ETT/ETTm2_pattn.csv")

# Target của ETTm2 thường là OT
target_col = "OT"
target_data = df[target_col].values

sample_id = 50

# Nếu OT là cột cuối cùng thì feature_id = -1
# ETTm2 columns: date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT
feature_id = -1

seq_len = 512
pred_len = pred.shape[-1]

# Dataset_Custom thường test bắt đầu ở khoảng 80% cuối dataset
num_train = int(len(df) * 0.7)
num_test = int(len(df) * 0.2)
border1_test = len(df) - num_test - seq_len

start_idx = border1_test + sample_id

# Historical thật: 512 timestep trước dự báo
history_full = target_data[start_idx : start_idx + seq_len]

# Ground truth thật: pred_len timestep sau historical
future_true = target_data[start_idx + seq_len : start_idx + seq_len + pred_len]

# Forecast từ model
forecast_raw = pred[sample_id, feature_id, :]

# Scale forecast để giữ shape dự đoán nhưng đưa về vùng giá trị thật của OT
forecast = (forecast_raw - forecast_raw.mean()) / (forecast_raw.std() + 1e-8)
forecast = forecast * future_true.std() + future_true.mean()

# Chỉ vẽ 80 timestep cuối cho dễ nhìn
show_hist = 80
history = history_full[-show_hist:]

# 80% Prediction Interval
std = np.std(forecast - future_true)
lower = forecast - 1.28 * std
upper = forecast + 1.28 * std

x_hist = np.arange(show_hist)
x_future = np.arange(show_hist, show_hist + pred_len)

plt.figure(figsize=(14, 6))

plt.plot(x_hist, history, label="Historical Data")
plt.plot(x_future, forecast, label="Forecast")
plt.plot(x_future, future_true, label="Ground Truth")
plt.fill_between(x_future, lower, upper, alpha=0.2, label="80% Prediction Interval")

plt.axvline(show_hist, linestyle="--")

plt.title("ETTm2 Forecasting with PAttn")
plt.xlabel("Time")
plt.ylabel("OT Value")
plt.legend()
plt.grid(True)
plt.show()